# GMRES from scratch

## 0. The problem

We want to solve a linear system

$$
A x = b, \qquad A \in \mathbb{R}^{n \times n} \text{ nonsingular}, \quad b \in \mathbb{R}^n .
$$

The catch: **we are not given $A$ as a matrix.** We are only given a function that applies it to a vector,

$$
\texttt{matvec} : v \mapsto A v .
$$

One call of this function is a *matvec* (matrix-vector product). We write $v$ rather than $x$ because the solver will apply $A$ to many vectors that are not the solution. We can't read the entries $a_{ij}$, factor $A$, or form $A^{-1}$. All we can do is pick vectors $v$ and look at $Av$. For example, if

$$
A = \begin{pmatrix} 2 & 1 \\ 0 & 3 \end{pmatrix}, \qquad
v = \begin{pmatrix} 1 \\ 1 \end{pmatrix}, \qquad
A v = \begin{pmatrix} 3 \\ 3 \end{pmatrix},
$$

then the solver never sees the matrix on the left, only that the input $(1, 1)$ came back as $(3, 3)$.

**Why only matvecs.** In the simulation, $A$ is the Jacobian $J = \partial F / \partial u$ of the nonlinear residual of an implicit time step. The product $J v$ is a directional derivative and costs about one evaluation of $F$ (e.g. with `jax.jvp`). Forming $J$ itself would take $n$ of those, one per column, and a dense LU factorisation of it costs $O(n^3)$ work and $O(n^2)$ memory. A matvec with a stencil operator costs $O(\mathrm{nnz}) = O(n)$.

**What counts as a solution.** An iterative solver produces approximations $x_0 = 0,\ x_1,\ x_2,\ \dots$ We can't measure the error $x - x_k$, since that needs the answer, but the residual

$$
r_k = b - A x_k
$$

costs one matvec. So the goal is to make $\|r_k\|_2$ small, stopping once $\|r_k\|_2 \le \text{rtol} \cdot \|b\|_2$.

**Cost is counted in matvecs.** The question for every version of the solver in this notebook is how many matvecs it needs to reach a given residual.

Conventions for the rest of the notebook: NumPy in float64, and every solver takes a `matvec` function, never a matrix.

## 1. The bare idea: search the Krylov space (`gmres_v0`)

In this step we build GMRES in its most literal form: write down every vector the solver *could* build with matvecs, and pick the best one. It works on a small problem, and on a bigger well-behaved one. At the end of the step we'll run it on a problem with widely spread eigenvalues and watch it fail, which is what Step 2 fixes.

### 1.1 Which vectors can the solver build?

At the start we know nothing except $b$, so the first guess is

$$
x_0 = 0, \qquad r_0 = b - A \cdot 0 = b .
$$

The only tool we have is the matvec. Applying it to the one vector we have gives $Ab$. Applying it to that gives $A^2 b$, and so on. After $k-1$ matvecs, the vectors in hand are

$$
b,\ Ab,\ A^2 b,\ \dots,\ A^{k-1} b .
$$

Scaling and adding vectors costs no matvecs, so we can also build any **linear combination** of them. The set of all those combinations is called the **Krylov space**:

$$
\mathcal{K}_k = \operatorname{span}\{\, b,\ Ab,\ A^2 b,\ \dots,\ A^{k-1} b \,\} .
$$

This isn't an arbitrary choice of where to look for the answer. Under the matvec-only rule from Step 0, $\mathcal{K}_k$ is *everything* the solver can reach.

### The example matrix

We'll use a small 3×3, called `geo3`, so every number can be checked by hand:

$$
A = \begin{pmatrix} 3 & 1 & 0 \\ 0 & 2 & 1 \\ 1 & 0 & 2 \end{pmatrix}, \qquad b = (1, 1, 1) .
$$

There's nothing special about it: it's nonsymmetric and well conditioned (determinant 13). Because it's 3×3, the Krylov space can only be a line ($k=1$), a plane ($k=2$) or all of $\mathbb{R}^3$ ($k=3$).

We also wrap the matrix in a small class, `CountingMatvec`. The solver is only handed this object, never the matrix, so it really is limited to matvecs. The object counts its calls, which lets us check the cost of each step.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from tutorial_utils import add_figure_caption  # numbered caption under a figure

np.set_printoptions(precision=4, suppress=True)

# geo3: a generic nonsymmetric, well-conditioned 3x3 (determinant 13).
A_geo3 = np.array([
    [3.0, 1.0, 0.0],
    [0.0, 2.0, 1.0],
    [1.0, 0.0, 2.0],
])
b_geo3 = np.array([1.0, 1.0, 1.0])


class CountingMatvec:
    """The function v -> A v, plus a count of how many times it has been called.

    The solver only ever gets this object, never A itself. The count lets us
    check how many matvecs each version of the solver spends.
    """

    def __init__(self, A):
        self._A = A  # hidden from the solver; only used inside __call__
        self.count = 0

    def __call__(self, v):
        self.count += 1
        return self._A @ v

Let's build the Krylov vectors for `geo3` by hand. Each new vector is the matvec of the previous one, so two matvecs give us three vectors.

In [ ]:
matvec = CountingMatvec(A_geo3)
print(matvec._A)
b = b_geo3

krylov_1 = b                  # b        (free: we were given it)
krylov_2 = matvec(krylov_1)   # A b      (1 matvec)
krylov_3 = matvec(krylov_2)   # A^2 b    (1 more matvec)

print("b     =", krylov_1)
print("A b   =", krylov_2)
print("A^2 b =", krylov_3)
print("matvecs used so far:", matvec.count)

### 1.2 Krylov vectors are polynomials in $A$

Write a general element of $\mathcal{K}_k$ with coefficients $c_1, \dots, c_k$:

$$
x = c_1\, b + c_2\, A b + c_3\, A^2 b + \dots + c_k\, A^{k-1} b
  = \underbrace{\left(c_1 I + c_2 A + c_3 A^2 + \dots + c_k A^{k-1}\right)}_{p(A)} b .
$$

So every vector in the Krylov space is $p(A)\, b$ for some polynomial

$$
p(t) = c_1 + c_2 t + \dots + c_k t^{k-1}
$$

of degree less than $k$. **Searching $\mathcal{K}_k$ is the same as searching over polynomials of degree less than $k$**, and the vector $c = (c_1, \dots, c_k)$ holds the polynomial's coefficients. We'll come back to this view in Step 6, where it explains why GMRES is fast on some matrices and slow on others.

### Why the exact answer is in there eventually

Is $x = A^{-1} b$ in the Krylov space at all? Yes, by step $n$ at the latest. The **Cayley–Hamilton theorem** says every matrix satisfies its own characteristic polynomial. For a 3×3:

$$
\chi(t) = \det(tI - A) = t^3 + a_2 t^2 + a_1 t + a_0
\quad\Longrightarrow\quad
A^3 + a_2 A^2 + a_1 A + a_0 I = 0 .
$$

The constant term is $a_0 = \chi(0) = \det(-A) = -\det A$, which is nonzero because $A$ is nonsingular. Move it to the other side and factor out one $A$:

$$
A \left(A^2 + a_2 A + a_1 I\right) = -a_0 I
\quad\Longrightarrow\quad
A^{-1} = -\frac{1}{a_0}\left(A^2 + a_2 A + a_1 I\right) .
$$

Multiply by $b$:

$$
A^{-1} b = -\frac{1}{a_0}\left(A^2 b + a_2\, A b + a_1\, b\right) ,
$$

which is a combination of $b$, $Ab$ and $A^2 b$, so it's in $\mathcal{K}_3$. The same argument for an $n \times n$ matrix puts $A^{-1} b$ in $\mathcal{K}_n$.

Let's check it on `geo3`, building $A^{-1}b$ from the three Krylov vectors above. (`np.poly` and `np.linalg.solve` look at the matrix directly. That's fine for *checking* the solver, as long as the solver itself never does it.)

In [ ]:
# np.poly returns the coefficients of det(tI - A), highest power first: [1, a2, a1, a0].
_, a2, a1, a0 = np.poly(A_geo3)
print(f"characteristic polynomial: t^3 + ({a2:.4g}) t^2 + ({a1:.4g}) t + ({a0:.4g})")

# A^{-1} b as a combination of the Krylov vectors b, A b, A^2 b.
x_from_krylov = -(krylov_3 + a2 * krylov_2 + a1 * krylov_1) / a0
x_exact = np.linalg.solve(A_geo3, b_geo3)  # checking only

print("x from the Krylov vectors:", x_from_krylov)
print("x from np.linalg.solve:   ", x_exact)

Reaching the answer at step $n$ is the **worst case**. For $n$ in the millions, $n$ steps would be useless. What makes GMRES worth using is that on many matrices a *good* answer appears for $k \ll n$. So the real question is how to pick the best vector in $\mathcal{K}_k$ at each step.

### 1.3 The GMRES rule: the smallest residual in $\mathcal{K}_k$

At step $k$, GMRES picks the vector in $\mathcal{K}_k$ whose residual is smallest:

$$
x_k = \arg\min_{x \in \mathcal{K}_k} \|b - A x\|_2 .
$$

The name says this: **G**eneralized **M**inimal **RES**idual.

**Turning it into a least-squares problem.** Put the Krylov vectors in the columns of an $n \times k$ matrix

$$
K_k = \big[\, b \;\big|\; Ab \;\big|\; \cdots \;\big|\; A^{k-1} b \,\big] .
$$

Any $x \in \mathcal{K}_k$ is $x = K_k c$ for some $c \in \mathbb{R}^k$, since multiplying a matrix by a vector *is* taking a combination of its columns. Then $Ax = (A K_k)\, c$, and the GMRES rule becomes

$$
\min_{c \in \mathbb{R}^k} \big\| b - (A K_k)\, c \big\|_2 .
$$

This is an ordinary **least-squares** problem with $n$ equations and $k$ unknowns. With $k < n$ there's usually no $c$ that makes the residual exactly zero, so we ask for the $c$ that makes it as small as possible. `np.linalg.lstsq` does exactly that.

A few points that are easy to mix up:

- **$c$ has $k$ entries**, one weight per Krylov vector. It says how much of $b$, $Ab$, … to mix in.
- **The guess is $x_k = K_k c$**, without the $A$. The product $(A K_k)\, c$ is $A x_k$, the thing compared against $b$.
- **$A K_k = [\,Ab \mid A^2 b \mid \cdots \mid A^k b\,]$ is $K_k$ shifted one column.** Its only new column is $A^k b$, so **each step costs one matvec**, and that same vector becomes the next column of $K$.
- **The residual is free.** $r_k = b - A x_k = b - (A K_k)\, c$, and we already have $A K_k$. No extra matvec is needed.
- **$c$ is re-solved from scratch at every step**, and all of its entries can change. What carries over from one step to the next is the basis $K$, not $c$.
- **The residual can never go up.** $\mathcal{K}_1 \subset \mathcal{K}_2 \subset \mathcal{K}_3 \subset \cdots$, so step $k+1$ searches everything step $k$ searched, and more. Its best can't be worse.

Let's do steps $k = 1$ and $k = 2$ by hand on `geo3`. We already have $b$, $Ab$ and $A^2 b$, so no new matvecs are needed yet.

In [ ]:
# ----- Step k = 1: search the line spanned by b -----
K_1 = np.column_stack([krylov_1])            # n x 1:  [b]
AK_1 = np.column_stack([krylov_2])           # n x 1:  [A b]

c_1, *_ = np.linalg.lstsq(AK_1, b, rcond=None)
x_1 = K_1 @ c_1                              # the guess
r_1 = b - AK_1 @ c_1                         # its residual, no new matvec

print("k = 1")
print("  c   =", c_1, "   ->  x_1 = c[0] * b")
print("  x_1 =", x_1)
print("  ||r_1|| =", np.linalg.norm(r_1))

# ----- Step k = 2: search the plane spanned by b and A b -----
K_2 = np.column_stack([krylov_1, krylov_2])  # n x 2:  [b, A b]
AK_2 = np.column_stack([krylov_2, krylov_3]) # n x 2:  [A b, A^2 b]

c_2, *_ = np.linalg.lstsq(AK_2, b, rcond=None)
x_2 = K_2 @ c_2
r_2 = b - AK_2 @ c_2

print("\nk = 2")
print("  c   =", c_2, "   ->  x_2 = c[0] * b + c[1] * A b")
print("  x_2 =", x_2)
print("  ||r_2|| =", np.linalg.norm(r_2))

Notice that the weight on $b$ went from about 0.29 at $k=1$ to about 0.71 at $k=2$. Adding $Ab$ to the mix changed the best weight on $b$ too, which is why $c$ has to be re-solved from scratch each step.

Now compare these against some other guesses. In particular, compare against "use only the highest power", $x = Ab$:

In [ ]:
candidates = {
    "x = 0  (do nothing)": np.zeros(3),
    "x = A b alone  (highest power at k = 2)": krylov_2,
    "best in K_1  (GMRES, k = 1)": x_1,
    "best in K_2  (GMRES, k = 2)": x_2,
}

for name, x in candidates.items():
    residual_norm = np.linalg.norm(b_geo3 - A_geo3 @ x)  # checking only
    print(f"{name:42s}  ||b - A x|| = {residual_norm:8.4f}")

Using only the highest power, $Ab$, is about ten times *worse* than doing nothing. Allowing polynomials of higher degree only helps when the coefficients are chosen well, and that's the job of the least-squares solve. Meanwhile the GMRES residuals go $1.73 \to 0.243 \to 0.093$, and at $k = 3$ they'll reach 0.

### 1.4 What the least-squares solution looks like geometrically

As $c$ varies, $(AK_k)\,c$ sweeps out the subspace $A\mathcal{K}_k$, spanned by the columns of $AK_k$. We want the point of that subspace closest to $b$. The closest point to $b$ in a subspace is its **orthogonal projection**, so what's left over, the residual, is perpendicular to the subspace:

$$
r_k \perp A\mathcal{K}_k
\qquad\Longleftrightarrow\qquad
(A K_k)^\mathsf{T}\, r_k = 0 .
$$

(These are the *normal equations* of least squares.) In 3D at $k = 2$: $A\mathcal{K}_2$ is a plane, $(AK_2)\,c$ is the foot of the perpendicular from $b$ down to that plane, and $r_2$ is the perpendicular itself.

Check it: every column of $AK_k$ should have a zero dot product with $r_k$, up to round-off.

In [ ]:
print("(A K_1)^T r_1 =", AK_1.T @ r_1)
print("(A K_2)^T r_2 =", AK_2.T @ r_2)

### 1.5 Putting it in a loop: `gmres_v0`

The by-hand steps above, repeated until the residual is small enough:

1. **$k = 0$:** set $x_0 = 0$, so $r_0 = b$. $K$ has no columns yet.
2. **For $k = 1, 2, 3, \dots$:**
   1. Add the next Krylov vector to $K$, and its matvec to $AK$. *(1 matvec.)*
   2. Solve $\min_c \|b - (AK_k)\,c\|_2$ for a $c$ of length $k$.
   3. Set $x_k = K_k c$ and $r_k = b - (AK_k)\,c$.
   4. Stop if $\|r_k\|_2 \le \text{rtol} \cdot \|b\|_2$.

The function takes a `matvec`, never a matrix. It returns the solution and a `record` dictionary describing the solve. Every later version of the solver (`gmres_v1`, `gmres_v2`, …) will keep this same signature and record so we can compare them directly.

It's written to be read, not to be fast. For example, it rebuilds $K$ and $AK$ from Python lists at every step and calls `lstsq` on the whole $n \times k$ problem every time.

In [ ]:
def gmres_v0(matvec, b, *, rtol=1e-10, max_iterations=None, verbose=False):
    """Bare GMRES: least squares over the literal Krylov basis [b, Ab, A^2 b, ...].

    Args:
        matvec: function v -> A v. The solver never sees A itself.
        b: right-hand side, shape (n,).
        rtol: stop once ||r_k|| <= rtol * ||b||.
        max_iterations: largest k to try. Defaults to n, where the exact
            solution must be in the Krylov space (Section 1.2).
        verbose: print c and the residual at every step.

    Returns:
        x: the last iterate x_k.
        record: dict with
            "iterations"        number of steps k taken,
            "matvecs"           number of matvec calls,
            "residual_history"  [||r_0||, ||r_1||, ..., ||r_k||],
            "converged"         whether the tolerance was reached.
    """
    n = b.shape[0]
    if max_iterations is None:
        max_iterations = n
    b_norm = np.linalg.norm(b)

    # ----- k = 0: start from x_0 = 0, so the residual is b itself. -----
    x = np.zeros_like(b)
    residual_history = [b_norm]
    matvec_count = 0
    converged = False

    krylov_vectors = []    # the columns of K_k:   b,   A b,   A^2 b, ...
    A_krylov_vectors = []  # the columns of A K_k: A b, A^2 b, A^3 b, ...
    next_krylov_vector = b

    for k in range(1, max_iterations + 1):
        # 1. Grow the basis by one column. This is the step's only matvec.
        krylov_vectors.append(next_krylov_vector)
        A_krylov_vectors.append(matvec(next_krylov_vector))
        matvec_count += 1
        # A^k b is both the new column of A K_k and the next column of K.
        next_krylov_vector = A_krylov_vectors[-1]

        K = np.column_stack(krylov_vectors)     # n x k
        AK = np.column_stack(A_krylov_vectors)  # n x k

        # 2. Find the weights c that make ||b - AK c|| as small as possible.
        #    Re-solved from scratch every step: all k entries can change.
        c, *_ = np.linalg.lstsq(AK, b, rcond=None)

        # 3. The guess, and its residual (free: we already have A K).
        x = K @ c
        r = b - AK @ c
        residual_history.append(np.linalg.norm(r))

        if verbose:
            print(f"k = {k}:  c = {c},  ||r_k|| = {residual_history[-1]:.4e}")

        # 4. Stop once the residual is small enough.
        if residual_history[-1] <= rtol * b_norm:
            converged = True
            break

    record = {
        "iterations": k,
        "matvecs": matvec_count,
        "residual_history": residual_history,
        "converged": converged,
    }
    return x, record

Run it on `geo3`. We expect it to reach the exact solution at $k = 3$, the size of the matrix, with exactly one matvec per step.

In [ ]:
matvec = CountingMatvec(A_geo3)
x, record = gmres_v0(matvec, b_geo3, verbose=True)

print()
print("x               =", x)
print("np.linalg.solve =", np.linalg.solve(A_geo3, b_geo3))
print("record:", {key: value for key, value in record.items() if key != "residual_history"})
print("residual history:", np.array(record["residual_history"]))
print("matvecs counted by the wrapper:", matvec.count)

That matches the by-hand steps: the same $c$ at $k = 1$ and $k = 2$, residuals $1.73 \to 0.243 \to 0.093 \to {\approx}\,0$, and three matvecs for three steps.

So the bare idea works on the 3×3. Next we'll try it on a much bigger problem.

### 1.6 A bigger problem

The 3×3 needed all $n = 3$ steps, the worst case from 1.2. GMRES is worth using because on many matrices it needs far fewer than $n$. Try `diag_narrow`, a 100×100 diagonal matrix with eigenvalues spread evenly between 1 and 1.5, and $b = (1, \dots, 1)$:

$$
A = \operatorname{diag}(\lambda_1, \dots, \lambda_{100}), \qquad
\lambda_i = 1 + \frac{0.5\,(i-1)}{99} \in [1,\ 1.5], \qquad
b = (1, 1, \dots, 1) .
$$

It's diagonal, so we can check the answer directly ($x_i = 1/\lambda_i$), but the solver still only gets matvecs. We keep the default tolerance, $\text{rtol} = 10^{-10}$.

In [ ]:
n_narrow = 100
A_diag_narrow = np.diag(np.linspace(1, 1.5, n_narrow))
b_diag_narrow = np.ones(n_narrow)

matvec = CountingMatvec(A_diag_narrow)
x_narrow, record_narrow = gmres_v0(matvec, b_diag_narrow)

relative_residuals_narrow = np.array(record_narrow["residual_history"]) / np.linalg.norm(b_diag_narrow)
print("converged:", record_narrow["converged"], "  iterations:", record_narrow["iterations"],
      "  matvecs:", matvec.count)
print("||r_k|| / ||b|| for k = 0, 1, ...:")
print(" ".join(f"{residual:.1e}" for residual in relative_residuals_narrow))

x_exact_narrow = np.linalg.solve(A_diag_narrow, b_diag_narrow)  # checking only
print("\nlargest error |x - x_exact|:", np.max(np.abs(x_narrow - x_exact_narrow)))

In [ ]:
steps_narrow = np.arange(0, record_narrow["iterations"] + 1)
rtol = 1e-10  # gmres_v0's default

line_color = "#2a78d6"
reference_color = "#52514e"

fig, ax = plt.subplots(figsize=(6.5, 4), constrained_layout=True)

ax.semilogy(steps_narrow, relative_residuals_narrow, color=line_color, linewidth=2, marker="o", markersize=5)
ax.axhline(rtol, color=reference_color, linewidth=1)
ax.text(0.2, rtol * 2, "rtol: stop once below this line", color=reference_color, fontsize=9, ha="left", va="bottom")
ax.set_xlabel("step k  (one matvec per step)")
ax.set_ylabel(r"$\|r_k\|_2 \,/\, \|b\|_2$")
ax.set_title(f"gmres_v0 residual on diag_narrow (n = {n_narrow})")

ax.set_xticks(steps_narrow)
ax.grid(True, which="major", color="#e4e3df", linewidth=1)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)

add_figure_caption(fig, "1.1", "gmres_v0 on diag_narrow (n = 100): the residual falls by about a factor of 10 per step and passes rtol after 11 steps.")
plt.show()

It converges in **11 steps, so 11 matvecs for 100 unknowns**, well short of the $n = 100$ that the worst case allows. The residual falls at every step, as 1.3 says it must, and the solution matches `np.linalg.solve`.

On the log scale of Figure 1.1 the curve is close to a straight line: each step cuts the residual by about a factor of 10. That factor depends on where the eigenvalues sit, not on $n$. Here they're bunched in $[1, 1.5]$. Step 6 explains where the factor comes from.

So a bigger $n$ doesn't have to mean more work. The next section goes the other way: a *smaller* matrix whose eigenvalues are spread far apart.

### 1.7 Where it breaks

Now take `diag_spread`, a 20×20 diagonal matrix, smaller than `diag_narrow`, but with eigenvalues spread evenly on a log scale from 1 to $10^4$ instead of bunched in $[1, 1.5]$. Again $b = (1, \dots, 1)$:

$$
A = \operatorname{diag}(\lambda_1, \dots, \lambda_{20}), \qquad
\lambda_i = 10^{4(i-1)/19} \in [1,\ 10^4], \qquad
b = (1, 1, \dots, 1) .
$$

It's diagonal, so its eigenvalues and eigenvectors are known exactly and nothing about it is hidden. The theory from 1.2 and 1.3 says:

- the residual **never increases**, and
- the residual is **exactly 0 by step $k = 20$**, since $A^{-1}b \in \mathcal{K}_{20}$.

Let's see whether `gmres_v0` agrees.

In [ ]:
n_spread = 20
A_diag_spread = np.diag(np.logspace(0, 4, n_spread))
b_diag_spread = np.ones(n_spread)

matvec = CountingMatvec(A_diag_spread)
x_spread, record_spread = gmres_v0(matvec, b_diag_spread)

relative_residuals = np.array(record_spread["residual_history"]) / np.linalg.norm(b_diag_spread)
print("converged:", record_spread["converged"], "  iterations:", record_spread["iterations"])
print("||r_k|| / ||b|| for k = 0, 1, ..., 20:")
print(relative_residuals)
print("\nsteps where the residual went UP:",
      [k for k in range(1, len(relative_residuals)) if relative_residuals[k] > relative_residuals[k - 1]])

Both predictions fail. The residual falls for four steps, then goes *up* at almost every step from $k = 5$ on, which the nested-subspace argument says can't happen. At $k = 20$ it's about 0.95 rather than 0. The math hasn't changed, so the problem is in the floating-point computation.

**Why: the Krylov vectors all turn toward the same direction.** For a diagonal matrix, entry $i$ of $A^j b$ is $\lambda_i^j$. The largest eigenvalue is $10^4$ and the next largest is about $6 \times 10^3$, so with each power the last entry pulls further ahead of the rest. After a few matvecs, every new Krylov vector points almost exactly along the last coordinate axis, the eigenvector of the largest eigenvalue. (This is **power iteration**, the standard way to *find* that eigenvector.) Below, the angle between each Krylov vector and the previous one shrinks toward zero.

In [ ]:
def krylov_matrix(matvec, b, k):
    """The literal Krylov basis K_k = [b, Ab, ..., A^{k-1} b] as an n x k array."""
    columns = [b]
    for _ in range(k - 1):
        columns.append(matvec(columns[-1]))
    return np.column_stack(columns)


def angle_degrees(u, v):
    """The angle between two vectors, in degrees."""
    cosine = abs(u @ v) / (np.linalg.norm(u) * np.linalg.norm(v))
    return np.degrees(np.arccos(min(cosine, 1.0)))


K_spread = krylov_matrix(CountingMatvec(A_diag_spread), b_diag_spread, n_spread)

for j in range(1, 8):
    print(f"angle between A^{j - 1} b and A^{j} b: {angle_degrees(K_spread[:, j - 1], K_spread[:, j]):8.4f} degrees")

When the columns of $K_k$ are nearly parallel, $K_k$ is nearly **rank deficient**: the information that distinguishes one column from another sits in the tiny components, far below the big shared direction. The **condition number** $\operatorname{cond}(K_k)$ measures this. Roughly, a least-squares solve with a matrix of condition number $\kappa$ loses $\log_{10}\kappa$ digits.

Double precision carries about 16 digits ($1/\varepsilon_{\text{machine}} \approx 4.5 \times 10^{15}$). Once $\operatorname{cond}(K_k)$ passes that, the new columns are, as far as the computer can tell, copies of the old ones. Adding them gives `lstsq` nothing new to work with, and the solve is dominated by round-off.

Figure 1.2 puts the two side by side: the residual of `gmres_v0` (left, on a linear scale so the rise is visible) and the condition number of its basis (right). Past $1/\varepsilon_{\text{machine}}$ the computed condition numbers are themselves mostly round-off. The only thing that matters is that they're above the line.

In [ ]:
condition_numbers = [np.linalg.cond(K_spread[:, :k]) for k in range(1, n_spread + 1)]
steps = np.arange(0, n_spread + 1)

fig, (ax_residual, ax_condition) = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)

ax_residual.plot(steps, relative_residuals, color=line_color, linewidth=2, marker="o", markersize=5)
ax_residual.plot([n_spread], [0.0], marker="o", markersize=8, color=reference_color)
ax_residual.annotate("exact arithmetic:\n0 at k = 20", xy=(n_spread, 0.0), xytext=(13.5, 0.25),
                     color=reference_color, fontsize=9, arrowprops=dict(arrowstyle="->", color=reference_color))
ax_residual.set_xlabel("step k")
ax_residual.set_ylabel(r"$\|r_k\|_2 \,/\, \|b\|_2$")
ax_residual.set_title("gmres_v0 residual on diag_spread (n = 20)")
ax_residual.set_ylim(-0.03, 1.05)

ax_condition.semilogy(np.arange(1, n_spread + 1), condition_numbers, color=line_color, linewidth=2, marker="o", markersize=5)
ax_condition.axhline(1 / np.finfo(float).eps, color=reference_color, linewidth=1)
ax_condition.text(n_spread, 1 / np.finfo(float).eps / 1e3, r"$1/\varepsilon_{\mathrm{machine}}$: no digits left",
                  color=reference_color, fontsize=9, ha="right", va="top")
ax_condition.set_xlabel("step k")
ax_condition.set_ylabel(r"cond$(K_k)$")
ax_condition.set_title("condition number of the Krylov basis")

for ax in (ax_residual, ax_condition):
    ax.set_xticks(range(0, n_spread + 1, 2))
    ax.grid(True, which="major", color="#e4e3df", linewidth=1)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)

add_figure_caption(fig, "1.2", r"gmres_v0 on diag_spread (n = 20). Left: the residual rises from step 5 instead of reaching 0. Right: cond$(K_k)$ hits $1/\varepsilon_{\mathrm{machine}}$ there.")
plt.show()

The two panels of Figure 1.2 line up. For $k \le 4$, while $\operatorname{cond}(K_k)$ is well below $1/\varepsilon_{\text{machine}}$, the residual falls as it should. At $k = 5$ the condition number reaches about $3 \times 10^{15}$, essentially at the line, and from there the residual creeps back up to about 0.95 and stays there.

### Summary of Step 1

- The solver can only reach the **Krylov space** $\mathcal{K}_k = \operatorname{span}\{b, Ab, \dots, A^{k-1}b\}$. Its elements are $p(A)\,b$ for polynomials $p$ of degree less than $k$.
- **GMRES** picks $x_k = K_k c$, with $c$ solving the least-squares problem $\min_c \|b - (AK_k)\,c\|_2$. Each step costs one matvec, and the residual can never increase.
- The exact solution is in $\mathcal{K}_n$ (Cayley–Hamilton), so in exact arithmetic GMRES finishes in at most $n$ steps.
- On a matrix with its eigenvalues bunched together it needs far fewer: 11 steps for the 100×100 `diag_narrow`.
- **But the literal basis $b, Ab, A^2b, \dots$ is unusable in floating point.** Its columns turn toward the dominant eigenvector, $\operatorname{cond}(K_k)$ grows exponentially, and after a handful of steps the least-squares solve is mostly round-off.

The search space $\mathcal{K}_k$ is fine. The problem is the basis we used to describe it. **Step 2** keeps exactly the same Krylov space but builds an *orthonormal* basis for it, one vector at a time (the Arnoldi process). In exact arithmetic it gives the same iterates as `gmres_v0`, but its basis stays well conditioned.